# 05 · Future climate (2041–2060) and soil

**Field Shift** (NASA Space Apps 2026), data notebook 5 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.* Needs notebook 01's files; uses 03's results for the demo field if they're there.

| Output | What's in it |
|---|---|
| `data/field.json` (adds `future` and `soil`) | The demo field: how hot days at flowering, frost and rain change by 2041–2060 under two emissions paths, model by model; soil texture, organic carbon, pH and water-holding capacity |
| `data/cells_future.json` | For every farmland square: the change by 2041–2060 under SSP2-4.5 and SSP5-8.5 (low / typical / high across climate models) |
| `data/cells_soil.json` | For every farmland square: topsoil (0–30 cm) clay, sand, silt, organic carbon, pH, bulk density, stones, nitrogen and CEC, plus water-holding capacity of the top metre (low / typical / high across the square) |

- **Future climate:** NASA NEX-GDDP-CMIP6, about 30 climate models downscaled to 25 km. For each model the notebook compares 2041–2060 with the model's own 2001–2020 (its historical run to 2014, then SSP2-4.5), the same years as today's numbers from notebook 03, then adds that change to today's numbers (the "delta" method). That way each model's own biases cancel out.
- **Soil:** SoilGrids 2.0 (ISRIC, 250 m), a mapped prediction, so the app should say "check with a soil test".

**How to run (about 10 minutes):** **Runtime → Run all** (your project ID is already filled in).

In [ ]:
EE_PROJECT = "skilled-clover-514221-p6"    # Earth Engine Cloud project (change it if someone else runs this)
DRIVE_FOLDER = "field-shift"            # folder for this notebook's files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: you pick the input files with an upload button. True: files are shared through Google Drive
REPO = ""                               # optional, e.g. "team-owner/field-shift": used only if a file isn't in Drive
OPTIONAL_INPUTS = ["field_weather.json"]    # from notebook 03: lets this notebook project today's numbers forward

## Setup

In [ ]:
import sys, subprocess
try:
    import pyproj, shapely                  # normally already in Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj", "shapely"], check=True)

import json, math, re, time, random, hashlib, urllib.request, datetime as dt
import concurrent.futures as cf
from pathlib import Path

import ee
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import shape

if EE_PROJECT == "your-cloud-project-id":
    raise ValueError("Put your Earth Engine Cloud project ID into EE_PROJECT (first code cell), then run again.")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            f"Google Drive didn't connect ({str(e)[:80]}).\n"
            "Run this cell again. In the Google window, pick the same account and tick 'Select all' "
            "(the Google Drive box) before pressing Continue.\n"
            "If Drive can't be used at all, set USE_DRIVE = False in the first cell. The notebook will then ask you to "
            "upload notebook 01's files, and you upload its downloads to the repo yourself.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA, CACHE = BASE / "data", BASE / "cache"
for folder in (DATA, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE)

TO_LAEA = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
CELL = 1000
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
BASELINE = (2001, 2020)          # the "normal" period: at least 15 years, printed in every file

def with_retries(fn, tries=6):
    """Earth Engine sometimes answers 'Too many requests' or times out: wait, then try again."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print("  retrying after:", str(e)[:100])
            time.sleep(min(60, 2 ** k + random.random()))

CHECKS = []
def check(ok, text, warn=False, fail=None):
    """Record and print a check: ✅ passed, ⚠️ worth a look, ❌ needs fixing (fail = what to print if it didn't pass)."""
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

def num(v, nd):
    """Round a value for the JSON files; empty or missing values become null."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return int(round(v)) if nd == 0 else round(v, nd)

def lth(values, nd, min_n=15):
    """[low, typical, high] = 10th percentile, median, 90th percentile; null if fewer than min_n values."""
    v = pd.Series(values, dtype="float64").dropna()
    if len(v) < min_n:
        return [None, None, None]
    return [num(v.quantile(0.1), nd), num(v.median(), nd), num(v.quantile(0.9), nd)]

INPUTS = ["grid.geojson", "field.json"] + list(globals().get("OPTIONAL_INPUTS", []))
missing = [n for n in ["grid.geojson", "field.json"] if not (DATA / n).exists()]
if missing and IN_COLAB and not REPO:
    from google.colab import files
    print("Choose these files (downloaded from earlier notebooks) in the upload box below, all at once:")
    print("   " + ", ".join(INPUTS) + ("   (the last ones only if you have them)" if len(INPUTS) > 2 else ""))
    for fname, content in files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)        # "field (1).json" -> "field.json"
        (DATA / clean).write_bytes(content)
        print("   got", clean)

def load_optional(name):
    """A data file if it's in the working folder, else an empty dict (no questions asked)."""
    path = DATA / name
    return json.loads(path.read_text()) if path.exists() else {}

def write_field_section(part, sections, sources):
    """This notebook's part of the demo field goes in its own file, field_<part>.json; notebook 06 merges them into field.json."""
    doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                    "field_key": FIELD_KEY, "sources": sources}}
    doc.update(sections)
    return write_json(DATA / f"field_{part}.json", doc)

def load_data_file(name):
    """A data file from the working folder, or else from the repo on GitHub (if REPO is set)."""
    path = DATA / name
    if path.exists():
        return json.loads(path.read_text())
    if REPO:
        url = f"https://raw.githubusercontent.com/{REPO}/main/data/{name}"
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode()
        path.write_text(text)
        return json.loads(text)
    if IN_COLAB:
        from google.colab import files
        print(f"{name} isn't in {DATA}. Choose it in the upload box below (you can pick several of notebook 01's files at once).")
        for fname, content in files.upload().items():
            clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)       # "field (1).json" -> "field.json"
            (DATA / clean).write_bytes(content)
        if path.exists():
            return json.loads(path.read_text())
    raise FileNotFoundError(f"{name} isn't in {DATA}. Run notebook 01 first (same Google account and DRIVE_FOLDER), "
                            "or set REPO so the file can be fetched from GitHub.")

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False))
    return path.stat().st_size / 1e6

GRID = load_data_file("grid.geojson")
FIELD_DOC = load_data_file("field.json")
FIELD = FIELD_DOC["field"]
SQUARES = [f["properties"] for f in GRID["features"]]
for sq in SQUARES:            # lower-left corner in metres, read from the EEA code, e.g. 1kmE4836N1990
    e, n = re.match(r"1kmE(\d+)N(\d+)", sq["id"]).groups()
    sq["x0"], sq["y0"] = int(e) * 1000, int(n) * 1000
FARM = [sq for sq in SQUARES if sq.get("farm")]
centre = shape(FIELD["outline"]).centroid                 # the field's data are worked out at the centre of its outline
FIELD_LON, FIELD_LAT = round(centre.x, 6), round(centre.y, 6)
FIELD_KEY = hashlib.md5(json.dumps(FIELD["outline"], sort_keys=True).encode()).hexdigest()[:8]   # changes if the outline is redrawn
FIELD_PT = ee.Geometry.Point([FIELD_LON, FIELD_LAT])
FIELD_GEOM = ee.Geometry(FIELD["outline"])
FIELD_AREA_M2 = FIELD["area_ha"] * 1e4

WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
CLC = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
RAINFED = WC.eq(40).And(CLC.eq(211))          # the project's rain-fed arable mask (10 m)

def square(sq):
    return ee.Geometry.Rectangle([sq["x0"], sq["y0"], sq["x0"] + CELL, sq["y0"] + CELL], "EPSG:3035", False)

def squares_fc(batch):
    return ee.FeatureCollection([ee.Feature(square(sq), {"id": sq["id"]}) for sq in batch])

print(f"{len(SQUARES):,} grid squares ({len(FARM):,} farmland); demo field in {FIELD['square']} "
      f"({FIELD['area_ha']} ha, outline {FIELD['outline_source'].split(':')[0]})")

## 1. Settings

In [ ]:
NOTEBOOK = "notebooks/05_future_soil.ipynb"
TODAY = dt.date.today()
G = ee.ImageCollection("NASA/GDDP-CMIP6")
SCENARIOS = {"historical": (2001, 2020), "ssp245": (2041, 2060), "ssp585": (2041, 2060)}
HIST_END = 2014                  # the models' historical runs stop in 2014: 2015-2020 of the baseline come from SSP2-4.5
NO_TEMP = {"CESM2", "CESM2-WACCM", "IITM-ESM", "TaiESM1"}          # models without daily max or min temperature
HOT_K, FROST_K = 303.15, 273.15                                    # 30 °C and 0 °C
IND = {"hot": "hot days (30 °C or more) from 15 April to 15 June", "tmaxfl": "average daytime high 15 April-15 June, °C",
       "frost": "frost nights December-February", "late": "frost nights March-April",
       "rainse": "October-May rain, mm", "rainau": "October-November rain, mm"}
PROJ = {"gddp": G.first().select("tasmax").projection()}

SOURCES = {
    "nex_gddp_cmip6": {"name": "NASA NEX-GDDP-CMIP6 daily downscaled climate projections", "ee_id": "NASA/GDDP-CMIP6", "res_m": 27830,
                       "licence": "CC0 1.0 (blanket licence for all NEX-GDDP-CMIP6 outputs since September 2022)",
                       "cite": "Thrasher, B. et al. (2022). NASA Global Daily Downscaled Projections, CMIP6. Scientific Data 9: 262. doi:10.1038/s41597-022-01393-4"},
    "soilgrids": {"name": "SoilGrids 2.0 (ISRIC), 250 m soil property predictions", "ee_id": "projects/soilgrids-isric/*_mean", "res_m": 250,
                  "licence": "CC BY 4.0", "cite": "Poggio, L. et al. (2021). SoilGrids 2.0. SOIL 7: 217-240. doi:10.5194/soil-7-217-2021"},
    "soilgrids_water": {"name": "SoilGrids 2.0 volumetric water content at 33 and 1500 kPa", "ee_id": "ISRIC/SoilGrids250m/v2_0", "res_m": 250,
                        "licence": "CC BY 4.0", "cite": "Turek, M. E. et al. (2023). International Soil and Water Conservation Research 11(2): 225-239"},
}

def models_for(scen, day):
    nxt = (dt.date.fromisoformat(day) + dt.timedelta(days=1)).isoformat()
    return set(with_retries(lambda: G.filterDate(day, nxt).filter(ee.Filter.eq("scenario", scen)).aggregate_array("model").getInfo()))

MODELS = sorted((models_for("historical", "2000-06-01") & models_for("ssp245", "2050-06-01") & models_for("ssp585", "2050-06-01")) - NO_TEMP)
check(len(MODELS) >= 20, f"{len(MODELS)} climate models with both scenarios: {', '.join(MODELS)}")

## 2. Future climate, model by model
One request per model: six numbers for 2001–2020 (the same years as today's numbers), and for 2041–2060 under SSP2-4.5 (middle path) and SSP5-8.5 (high emissions), at each 25 km model pixel over the basin. Each model is saved to Drive as it finishes.

In [ ]:
def pixel_map(points):
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([p["lon"], p["lat"]]), {"id": p["id"]}) for p in points])
    ll = ee.Image.pixelLonLat().reproject(PROJ["gddp"])
    fc = ll.sampleRegions(collection=pts, projection=PROJ["gddp"], geometries=False)
    return {f["properties"]["id"]: (round(f["properties"]["longitude"], 4), round(f["properties"]["latitude"], 4))
            for f in with_retries(lambda: fc.getInfo())["features"]}

pix_file = CACHE / f"05_pixels_gddp_{FIELD_KEY}.json"
if pix_file.exists():
    PIX = {k: tuple(v) for k, v in json.loads(pix_file.read_text()).items()}
else:
    PIX = pixel_map(FARM + [{"id": "field", "lon": FIELD_LON, "lat": FIELD_LAT}])
    pix_file.write_text(json.dumps(PIX))
UNIQUE = sorted(set(PIX.values()))
PIX_FC = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"id": f"{lon},{lat}"}) for lon, lat in UNIQUE])
print(f"{len(UNIQUE)} model pixels cover the farmland squares")

def indicators(model, scen):
    y0, y1 = SCENARIOS[scen]
    runs = G.filter(ee.Filter.eq("model", model))
    if scen == "historical":       # 2001-2014 from the historical run, 2015-2020 from SSP2-4.5 (the standard way to extend it)
        col = (runs.filter(ee.Filter.eq("scenario", "historical")).filterDate(f"{y0}-01-01", f"{HIST_END + 1}-01-01")
               .merge(runs.filter(ee.Filter.eq("scenario", "ssp245")).filterDate(f"{HIST_END + 1}-01-01", f"{y1 + 1}-01-01")))
    else:
        col = runs.filter(ee.Filter.eq("scenario", scen)).filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
    if model == "GFDL-CM4":
        col = col.filter(ee.Filter.eq("grid_label", "gr1"))          # this model comes in two grids: use one
    n = y1 - y0 + 1
    flower = col.filter(ee.Filter.dayOfYear(105, 166))               # about 15 April to 15 June
    winter = col.filter(ee.Filter.calendarRange(12, 2, "month"))
    spring = col.filter(ee.Filter.calendarRange(3, 4, "month"))
    season = col.filter(ee.Filter.calendarRange(10, 5, "month"))
    autumn = col.filter(ee.Filter.calendarRange(10, 11, "month"))
    bands = {
        "hot": flower.map(lambda i: i.select("tasmax").gte(HOT_K)).sum().divide(n),
        "tmaxfl": flower.select("tasmax").mean().subtract(273.15),
        "frost": winter.map(lambda i: i.select("tasmin").lte(FROST_K)).sum().divide(n),
        "late": spring.map(lambda i: i.select("tasmin").lte(FROST_K)).sum().divide(n),
        "rainse": season.select("pr").sum().multiply(86400).divide(n),
        "rainau": autumn.select("pr").sum().multiply(86400).divide(n),
    }
    return [img.rename(f"{scen}_{k}") for k, img in bands.items()]

def fetch_model(model):
    path = CACHE / f"05_gddp_b2001_{model}.csv"
    if path.exists():
        df = pd.read_csv(path, index_col="id")
        if set(df.index) >= {f"{lon},{lat}" for lon, lat in UNIQUE}:
            return df
    img = ee.Image.cat([b for scen in SCENARIOS for b in indicators(model, scen)])
    # period means have no projection of their own (Earth Engine would work at 1 degree), so work on the 0.25 degree model grid
    feats = with_retries(lambda: img.reduceRegions(collection=PIX_FC, reducer=ee.Reducer.first(), crs=PROJ["gddp"]).getInfo())["features"]
    df = pd.DataFrame([f["properties"] for f in feats]).set_index("id")
    df.to_csv(path)
    return df

t0, per_model = time.time(), {}
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(fetch_model, m): m for m in MODELS}
    for i, fut in enumerate(cf.as_completed(futs), 1):
        try:
            per_model[futs[fut]] = fut.result()
        except Exception as e:
            print(f"  {futs[fut]} failed and is left out: {str(e)[:100]}")
        if i % 5 == 0 or i == len(futs):
            print(f"  {i}/{len(futs)} models done, {time.time() - t0:.0f} s")
check(len(per_model) >= 20, f"Future climate from {len(per_model)} models")

## 3. Change by 2041–2060, across models
Temperatures and day counts: future minus past. Rain: percentage change. Low / typical / high = 10th percentile, median and 90th percentile across the models.

In [ ]:
def changes(pix):
    rows = []
    for model, df in per_model.items():
        if pix not in df.index:
            continue
        r = df.loc[pix]
        row = {"model": model}
        for scen in ("ssp245", "ssp585"):
            for k in IND:
                past, fut = r.get(f"historical_{k}"), r.get(f"{scen}_{k}")
                if past is None or fut is None or not np.isfinite(past) or not np.isfinite(fut):
                    row[f"{scen}_{k}"] = np.nan
                elif k.startswith("rain"):
                    row[f"{scen}_{k}"] = (fut / past - 1) * 100 if past > 0 else np.nan
                else:
                    row[f"{scen}_{k}"] = fut - past
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")

ND = {"hot": 1, "tmaxfl": 1, "frost": 1, "late": 1, "rainse": 0, "rainau": 0}
records, cells_out, change_tables = {}, {}, {}
for pix in UNIQUE:
    key = f"{pix[0]},{pix[1]}"
    ch = changes(key)
    change_tables[key] = ch
    records[f"g{len(records)}"] = {scen: {k: lth(ch[f"{scen}_{k}"], ND[k], min_n=10) for k in IND} for scen in ("ssp245", "ssp585")}
rec_of = {f"{p[0]},{p[1]}": f"g{i}" for i, p in enumerate(UNIQUE)}
for sq in FARM:
    cells_out[sq["id"]] = rec_of[f"{PIX[sq['id']][0]},{PIX[sq['id']][1]}"]
print(f"Future changes for {len(cells_out):,} squares from {len(records)} model pixels")

fkey = f"{PIX['field'][0]},{PIX['field'][1]}"
fch = change_tables[fkey]
FRESH_WEATHER = load_optional("field_weather.json").get("weather") or FIELD_DOC.get("weather") or {}   # 03's own file first: a merged field.json from 06 can hold an older weather section
weather = FRESH_WEATHER.get("typical", {})
if not weather:
    print("No field_weather.json from notebook 03, so the field gets changes only (no projected values).")
NOW_KEYS = {"hot": "hot_days_flower", "tmaxfl": "tmax_flower", "frost": "frost_days_winter", "late": "late_frost_days",
            "rainse": "rain_season", "rainau": "rain_autumn"}
future = {"made_by": f"{NOTEBOOK} ({RUN_AT})", "field_key": FIELD_KEY,
          "about": "Change by 2041-2060 against each model's 2001-2020 (historical run to 2014, then SSP2-4.5), added to "
                   "today's typical value for the same years from notebook 03 "
                   "(the delta method). Temperatures and counts: difference; rain: percentage change. "
                   "low / typical / high = 10th percentile, median and 90th percentile across the models.",
          "periods": {"past": "2001-2020 (historical run to 2014, then SSP2-4.5)", "future": "2041-2060"},
          "scenarios": {"ssp245": "SSP2-4.5, middle emissions path", "ssp585": "SSP5-8.5, high emissions path"},
          "indicators": IND, "models": list(fch.index), "change": {}, "projected": {}, "by_model": {}}
for scen in ("ssp245", "ssp585"):
    future["change"][scen] = {k: lth(fch[f"{scen}_{k}"], ND[k], min_n=10) for k in IND}
    future["by_model"][scen] = {k: {m: num(v, ND[k]) for m, v in fch[f"{scen}_{k}"].items()} for k in IND}
    proj = {}
    for k in IND:
        now = (weather.get(NOW_KEYS[k]) or [None, None, None])[1]
        lo, mid, hi = future["change"][scen][k]
        if now is None or mid is None:
            proj[k] = [None, None, None]
        elif k.startswith("rain"):
            proj[k] = [num(now * (1 + c / 100), 0) for c in (lo, mid, hi)]
        else:
            proj[k] = [num(max(0, now + c) if k != "tmaxfl" else now + c, ND[k]) for c in (lo, mid, hi)]
    future["projected"][scen] = proj
future["now"] = {k: weather.get(NOW_KEYS[k]) for k in IND}

# If notebook 03 corrected the field's temperatures with a nearby station, project those too (same model changes)
fix = FRESH_WEATHER.get("station_corrected")
if fix:
    now_fix = {k: fix["typical"].get(NOW_KEYS[k]) for k in ("hot", "tmaxfl", "frost", "late")}
    future["now_station_corrected"] = now_fix
    future["projected_station_corrected"] = {}
    for scen in ("ssp245", "ssp585"):
        proj = {}
        for k, now_v in now_fix.items():
            lo, mid, hi = future["change"][scen][k]
            n = (now_v or [None, None, None])[1]
            proj[k] = ([None] * 3 if n is None or mid is None
                       else [num(max(0, n + c) if k != "tmaxfl" else n + c, ND[k]) for c in (lo, mid, hi)])
        future["projected_station_corrected"][scen] = proj
    print("Station-corrected hot days at flowering at the field: now", now_fix["hot"],
          "| 2041-2060 SSP2-4.5", future["projected_station_corrected"]["ssp245"]["hot"])

c245, c585 = future["change"]["ssp245"], future["change"]["ssp585"]
check(c245["tmaxfl"][1] is not None and c245["tmaxfl"][1] > 0 and c585["tmaxfl"][1] >= c245["tmaxfl"][1],
      f"Warming at flowering by 2041-2060: +{c245['tmaxfl'][1]} °C (SSP2-4.5), +{c585['tmaxfl'][1]} °C (SSP5-8.5), median of models")
check(c585["hot"][1] is not None and c585["hot"][1] > 0,
      f"Hot days at flowering rise: +{c245['hot'][1]} (SSP2-4.5) and +{c585['hot'][1]} (SSP5-8.5) days a season", warn=True)
print("Change in October-May rain (%):", c245["rainse"], c585["rainse"])
if future["now"]["hot"]:
    print(f"Hot days at flowering at the field: now {future['now']['hot'][1]}, 2041-2060 {future['projected']['ssp245']['hot']} "
          f"(SSP2-4.5), {future['projected']['ssp585']['hot']} (SSP5-8.5)")

## 4. Soil (SoilGrids 2.0)
Topsoil (0–30 cm) as the thickness-weighted average of SoilGrids' 0–5, 5–15 and 15–30 cm layers. Water-holding capacity of the top metre = water held between field capacity (33 kPa) and wilting point (1500 kPa), added up over 0–100 cm. For every square, low / typical / high are the 10th, 50th and 90th percentile of the 250 m pixels on its rain-fed arable land, or of the whole square if that land is too scattered.

In [ ]:
# Nitrogen: the Earth Engine copy stores 10x ISRIC's cg/kg (mg/kg), so divide by 1000; checked against ISRIC's own service below
SOIL_PROPS = {"clay": 10, "sand": 10, "silt": 10, "soc": 10, "phh2o": 10, "bdod": 100, "cfvo": 10, "nitrogen": 1000, "cec": 10}
SOIL_UNITS = {"clay": "%", "sand": "%", "silt": "%", "soc": "g/kg", "phh2o": "pH", "bdod": "g/cm³", "cfvo": "% volume",
              "nitrogen": "g/kg", "cec": "cmol(c)/kg", "awc": "mm in the top metre"}
W30 = [5, 10, 15]                                                      # layer thicknesses 0-5, 5-15, 15-30 cm

def topsoil(prop):
    img = ee.Image(f"projects/soilgrids-isric/{prop}_mean").select([0, 1, 2])   # bands are the 6 depths, top first
    return img.multiply(ee.Image.constant(W30)).reduce(ee.Reducer.sum()).divide(sum(W30) * SOIL_PROPS[prop]).rename(f"soil_{prop}")

def awc():
    depths = {"0_5cm": 0.05, "5_15cm": 0.10, "15_30cm": 0.15, "30_60cm": 0.30, "60_100cm": 0.40}   # layer thickness, m
    fc_ = ee.Image("ISRIC/SoilGrids250m/v2_0/wv0033")
    wp = ee.Image("ISRIC/SoilGrids250m/v2_0/wv1500")
    # Read the layer near the field to learn its units: a fraction (0.3), percent (30) or 10^-3 cm3/cm3 (300, = mm per m)
    probe = with_retries(lambda: fc_.select("val_0_5cm_mean").reduceRegion(ee.Reducer.mean(), FIELD_PT.buffer(2000), 250).getInfo())
    v = num(list(probe.values())[0], 3) if probe else None
    per_m = 1 if v is None else (1000 if v <= 1.5 else (10 if v <= 70 else 1))      # multiplier to mm of water per m of soil
    check(v is not None, f"Water-retention layer near the field reads {v} at field capacity (0-5 cm): taken as "
          f"{ {1000: 'a fraction', 10: 'percent', 1: 'mm per metre'}[per_m] } of soil volume", warn=True,
          fail="Couldn't read the water-retention layer near the field: assuming mm per metre")
    total = None
    for d, t in depths.items():
        part = fc_.select(f"val_{d}_mean").subtract(wp.select(f"val_{d}_mean")).toFloat().multiply(t * per_m)
        total = part if total is None else total.add(part)
    return total.rename("soil_awc")                                    # mm of water the top metre holds for crops

SOIL = ee.Image.cat([topsoil(p) for p in SOIL_PROPS] + [awc()])
SPROJ = ee.Image("projects/soilgrids-isric/clay_mean").projection()
MASK250 = (RAINFED.unmask(0).reproject(WC.projection())
           .reduceResolution(ee.Reducer.mean(), maxPixels=1024).reproject(SPROJ).gte(0.5))
SOIL_KEYS = [f"soil_{p}" for p in SOIL_PROPS] + ["soil_awc"]

def soil_batch(args):
    batch, masked = args
    img = SOIL.updateMask(MASK250) if masked else SOIL
    out = img.reduceRegions(collection=squares_fc(batch), reducer=ee.Reducer.percentile([10, 50, 90]), scale=250, tileScale=4)
    return [f["properties"] for f in with_retries(lambda: out.getInfo())["features"]]

soil_file = CACHE / "05_soil_cells_v2.json"
if soil_file.exists():
    soil_rows = json.loads(soil_file.read_text())
else:
    batches = [FARM[i:i + 900] for i in range(0, len(FARM), 900)]
    with cf.ThreadPoolExecutor(max_workers=6) as pool:
        masked = list(pool.map(soil_batch, [(b, True) for b in batches]))
        whole = list(pool.map(soil_batch, [(b, False) for b in batches]))
    soil_rows = {"masked": [r for part in masked for r in part], "whole": [r for part in whole for r in part]}
    soil_file.write_text(json.dumps(soil_rows))
by_id = {k: {r["id"]: r for r in v} for k, v in soil_rows.items()}

ND_SOIL = {"clay": 0, "sand": 0, "silt": 0, "soc": 1, "phh2o": 1, "bdod": 2, "cfvo": 0, "nitrogen": 2, "cec": 0, "awc": 0}
soil_cells, from_whole = {}, 0
for sq in FARM:
    r = by_id["masked"].get(sq["id"], {})
    if r.get("soil_clay_p50") is None:
        r, from_whole = by_id["whole"].get(sq["id"], {}), from_whole + 1
    soil_cells[sq["id"]] = {k.replace("soil_", ""): [num(r.get(f"{k}_p{q}"), ND_SOIL[k.replace("soil_", "")]) for q in (10, 50, 90)]
                            for k in SOIL_KEYS}
filled = sum(1 for v in soil_cells.values() if v["clay"][1] is not None)
check(filled >= 0.95 * len(FARM), f"Soil for {filled:,} of {len(FARM):,} farmland squares ({from_whole} from the whole square)")

# the demo field: SoilGrids at the field's centre, plus the 90% prediction range from the SoilGrids web service if it answers
field_soil_raw = with_retries(lambda: SOIL.reduceRegion(ee.Reducer.first(), FIELD_PT, 250).getInfo())
soil = {"made_by": f"{NOTEBOOK} ({RUN_AT})", "field_key": FIELD_KEY,
        "about": "SoilGrids 2.0 prediction at the field's centre (250 m). Topsoil = 0-30 cm. Water-holding capacity = top metre. "
                 "A prediction, not a measurement: check with a soil test.",
        "units": SOIL_UNITS,
        "values": {k.replace("soil_", ""): num(field_soil_raw.get(k), ND_SOIL[k.replace("soil_", "")]) for k in SOIL_KEYS},
        "range_90pc": {}, "trial_reported": "clay loam, pH about 8.2, total nitrogen about 0.9 g/kg, CEC about 21.3 cmol(c)/kg (Denora et al. 2025, typed in from the paper)"}
try:
    url = ("https://rest.isric.org/soilgrids/v2.0/properties/query?" + "&".join(
        [f"lat={FIELD_LAT}", f"lon={FIELD_LON}"] + [f"property={p}" for p in ("clay", "sand", "soc", "phh2o")]
        + [f"depth={d}" for d in ("0-5cm", "5-15cm", "15-30cm")] + [f"value={v}" for v in ("Q0.05", "Q0.95")]))
    with urllib.request.urlopen(url, timeout=60) as r:
        layers = json.loads(r.read().decode())["properties"]["layers"]
    for layer in layers:
        f = layer.get("unit_measure", {}).get("d_factor") or 1
        vals = {q: sum(d["values"][q] * w for d, w in zip(layer["depths"], W30)) / sum(W30) / f for q in ("Q0.05", "Q0.95")}
        soil["range_90pc"][layer["name"]] = [round(vals["Q0.05"], 1), round(vals["Q0.95"], 1)]
except Exception as e:
    print("SoilGrids web service didn't answer, so the field's prediction range is left out:", str(e)[:80])

fs = soil["values"]
print("Demo field soil:", json.dumps(fs))
check(fs["clay"] is not None and 20 <= fs["clay"] <= 50 and fs["phh2o"] is not None and 7.3 <= fs["phh2o"] <= 8.7,
      f"Field soil close to the trial's report (clay loam, pH about 8.2): clay {fs['clay']}%, sand {fs['sand']}%, pH {fs['phh2o']}",
      warn=True)
check(fs["awc"] is not None and 60 <= fs["awc"] <= 300, f"Water-holding capacity of the top metre at the field: {fs['awc']} mm", warn=True)

# Nitrogen against ISRIC's own SoilGrids service at the field; if they disagree, nitrogen is left out rather than guessed
try:
    url = ("https://rest.isric.org/soilgrids/v2.0/properties/query?" + "&".join(
        [f"lat={FIELD_LAT}", f"lon={FIELD_LON}", "property=nitrogen", "value=mean"] + [f"depth={d}" for d in ("0-5cm", "5-15cm", "15-30cm")]))
    with urllib.request.urlopen(url, timeout=60) as r:
        layer = json.loads(r.read().decode())["properties"]["layers"][0]
    isric_n = sum(d["values"]["mean"] * w for d, w in zip(layer["depths"], W30)) / sum(W30) / (layer["unit_measure"].get("d_factor") or 100)
    ok = fs["nitrogen"] is not None and 0.67 <= fs["nitrogen"] / isric_n <= 1.5
    check(ok, f"Field nitrogen {fs['nitrogen']} g/kg matches ISRIC's own service ({isric_n:.2f} g/kg, 0-30 cm); the trial measured about 0.9 g/kg",
          fail=f"Field nitrogen {fs['nitrogen']} g/kg doesn't match ISRIC's own service ({isric_n:.2f} g/kg): nitrogen left out")
    if not ok:
        for v in soil_cells.values():
            v["nitrogen"] = [None, None, None]
        fs["nitrogen"] = None
except Exception as e:
    check(False, "Nitrogen checked against ISRIC", warn=True,
          fail=f"Couldn't reach ISRIC's service to check nitrogen ({str(e)[:60]}); kept, as it matched on 14 Nov")

## 5. Write the files

In [ ]:
FUT_IND = {k: {"what": v, "shape": "{scenario: [low, typical, high]} change across models"} for k, v in IND.items()}
future_doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                       "keys": "cells maps each farmland square id to a key in records; the values are records[cells[id]]",
                       "what": "change by 2041-2060 against 2001-2020 in each model (historical run to 2014, then SSP2-4.5; temperatures and day counts: difference; rain: % change). "
                               "Projected value for a square = its typical value in cells_weather.json + this change "
                               "(for rain: x (1 + change/100)).",
                       "low_typical_high": "10th percentile, median and 90th percentile across the climate models",
                       "scenarios": future["scenarios"], "models": sorted(per_model), "indicators": FUT_IND,
                       "sources": {"nex_gddp_cmip6": SOURCES["nex_gddp_cmip6"]}},
              "cells": cells_out, "records": records}
soil_doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                     "keys": "grid square ids from grid.geojson (farmland squares only)",
                     "low_typical_high": "10th, 50th and 90th percentile of the 250 m SoilGrids pixels in the square "
                                         "(its rain-fed arable land where possible)",
                     "topsoil": "0-30 cm, thickness-weighted", "units": SOIL_UNITS,
                     "sources": {k: SOURCES[k] for k in ("soilgrids", "soilgrids_water")}},
            "cells": soil_cells}

OUTPUTS = ["field_future_soil.json", "cells_future.json", "cells_soil.json"]
check(write_field_section("future_soil", {"future": future, "soil": soil}, SOURCES) < 5, "field_future_soil.json written")
for name, obj in (("cells_future.json", future_doc), ("cells_soil.json", soil_doc)):
    mb = write_json(DATA / name, obj)
    check(mb < 5, f"{name} written: {mb:.2f} MB")

## 6. Chart
The chart of hot days at flowering, today and by 2041–2060, is drawn by 06 from `field.json`, with the other charts for the pitch.

## 7. Checks, downloads and what to paste back

In [ ]:
print("\n".join(CHECKS))
bad = [c for c in CHECKS if c.startswith("❌")]
print("\nAll essential checks passed." if not bad else f"\n{len(bad)} check(s) failed: paste the lines above into the chat.")
if IN_COLAB:
    from google.colab import files
    for name in OUTPUTS:
        files.download(str(DATA / name))
    print("Downloading", ", ".join(OUTPUTS), "- upload them to the repo's data folder (replace the old ones).")

In [ ]:
print("=== Paste this back to Claude ===")
print(f"05_future_soil run {RUN_AT}; {len(per_model)} models; {len(UNIQUE)} model pixels")
print("field change ssp245:", json.dumps(future["change"]["ssp245"]))
print("field change ssp585:", json.dumps(future["change"]["ssp585"]))
print("field now / projected hot days:", future["now"]["hot"], future["projected"]["ssp245"]["hot"], future["projected"]["ssp585"]["hot"])
print("field soil:", json.dumps(soil["values"]), json.dumps(soil["range_90pc"]))
print("\n".join(CHECKS))